In [11]:
import polars as pl
import os
import glob
from datetime import datetime

# =====================================================================
# GLOBAL PATH CONFIGURATION
# =====================================================================
first_glob = os.path.expanduser("~").replace("\\", "/")
test_path = f"{first_glob}/Concentrix Corporation"

if not os.path.exists(test_path):
    raise FileNotFoundError(f"Root directory not found: {test_path}")

base_path = f"{test_path}/WFM-Expedia-HCM - Branding files"

folder_paths = {
    "roster_path": f"{base_path}/Schedule/Schedule (Ops version)/2026/Master_Schedule_Merged.xlsx",
    "ot_plan_path": f"{base_path}/OT/VN Expedia - OT Plan.xlsx",
    "ramco_ot_csv_path": f"{base_path}/BI_Task/CODE/Python_Code/ramco_ot.csv",
    "hc_parquet_path": f"{base_path}/BI_Task/CODE/Resources/hc_extend_combination.parquet",
    "rta_report_folder": f"{base_path}/Rawdata/STORAGE_OUTPUT_RTA",
    "rta_intervals_folder": f"{base_path}/Rawdata/OUTPUT_AGENT_ACTIVITY_INTERVALS",
    "output_ot_path": f"{base_path}/Rawdata/OUTPUT_OT_REGISTRATION_REPORT.xlsx"
}

roster_path = folder_paths["roster_path"]
ot_plan_path = folder_paths["ot_plan_path"]
ramco_ot_csv_path = folder_paths["ramco_ot_csv_path"]
hc_parquet_path = folder_paths["hc_parquet_path"]
rta_report_folder = folder_paths["rta_report_folder"]
rta_intervals_folder = folder_paths["rta_intervals_folder"]
output_ot_path = folder_paths["output_ot_path"]

# Define Holidays List
holidays_data = [
    ("2023-01-01", "New Year's Day"), ("2023-01-21", "TET"), ("2023-01-22", "TET"), ("2023-01-23", "TET"), ("2023-01-24", "TET"), ("2023-01-25", "TET"),
    ("2023-04-29", "KING'S DAY"), ("2023-04-30", "Reunification Day"), ("2023-05-01", "Labor Day"), ("2023-09-01", "Independence Day"), ("2023-09-02", "Independence Day"),
    ("2024-01-01", "New Year's Day"), ("2024-02-09", "TET"), ("2024-02-10", "TET"), ("2024-02-11", "TET"), ("2024-02-12", "TET"), ("2024-02-13", "TET"),
    ("2024-04-18", "KING'S DAY"), ("2024-04-30", "Reunification Day"), ("2024-05-01", "Labor Day"), ("2024-09-02", "Independence Day"), ("2024-09-03", "Independence Day"),
    ("2025-01-01", "New Year's Day"), ("2025-01-28", "TET"), ("2025-01-29", "TET"), ("2025-01-30", "TET"), ("2025-01-31", "TET"), ("2025-02-01", "TET"),
    ("2025-04-07", "KING'S DAY"), ("2025-04-30", "Reunification Day"), ("2025-05-01", "Labor Day"), ("2025-09-01", "Independence Day"), ("2025-09-02", "Independence Day"),
    ("2026-01-01", "New Year's Day"), ("2026-02-16", "TET"), ("2026-02-17", "TET"), ("2026-02-18", "TET"), ("2026-02-19", "TET"), ("2026-02-20", "TET"),
    ("2026-04-26", "KING'S DAY"), ("2026-04-30", "Reunification Day"), ("2026-05-01", "Labor Day"), ("2026-09-01", "Independence Day"), ("2026-09-02", "Independence Day")
]
holidays_df = pl.DataFrame(holidays_data, schema=["Date", "Holiday_Name"]).with_columns(pl.col("Date").str.strptime(pl.Date, "%Y-%m-%d"))

def mround(col_expr, multiple):
    return ((col_expr / multiple) + 1e-9).round(0) * multiple

print("STARTING RECONCILIATION REPORT PROCESSING (HOLIDAY AUTO-GEN INCLUDED)...")
print("-" * 60)

# =====================================================================
# [1/10] READ ROSTER, HC PARQUET & OT PLAN
# =====================================================================
print("[1/10] Processing Roster, HC Parquet, and OT Plan data...")
roster_df = pl.read_excel(roster_path, sheet_name="Sheet1").rename({"OracleID": "Emp ID", "Employee Name": "Agent Name"})

roster_meta_cols = [c for c in ["Emp ID", "Email", "Agent Name", "TL", "Team", "Status"] if c in roster_df.columns]
roster_meta = roster_df.select(roster_meta_cols).unique("Emp ID")

id_vars = [c for c in ["IEX ID", "IEX", "Emp ID", "Email", "Agent Name", "Status", "Team"] if c in roster_df.columns]
date_cols = [c for c in roster_df.columns if c not in id_vars]
roster_melted = roster_df.unpivot(index=id_vars, on=date_cols, variable_name="Scheduled_Date_Str", value_name="Shift")
roster_melted = roster_melted.with_columns([
    pl.col("Scheduled_Date_Str").str.slice(0, 10).str.strptime(pl.Date, "%Y-%m-%d", strict=False).alias("Scheduled_Date"),
    pl.when(pl.col("Shift") == "-").then(pl.lit("OFF")).otherwise(pl.col("Shift")).fill_null("OFF").alias("Shift")
])
roster_key = roster_melted.select(["Emp ID", "Scheduled_Date", "Shift"])

# Read Parquet to extract Designation and Supervisor globally
if os.path.exists(hc_parquet_path):
    hc_full = pl.read_parquet(hc_parquet_path)
    pq_cols = [c for c in ["Date", "OracleID", "Supervisor Name", "Designation", "LOB"] if c in hc_full.columns]
    hc_ext = hc_full.select(pq_cols).with_columns([
        pl.col("Date").cast(pl.String).str.slice(0, 10).str.strptime(pl.Date, "%Y-%m-%d", strict=False),
        pl.col("OracleID").cast(pl.Int64, strict=False)
    ])
else:
    hc_ext = pl.DataFrame({"Date": [], "OracleID": [], "Supervisor Name": [], "Designation": [], "LOB": []}).with_columns([pl.col("Date").cast(pl.Date), pl.col("OracleID").cast(pl.Int64)])

ot_df = pl.read_excel(ot_plan_path, sheet_name="OT Register").filter(pl.col("Emp ID").is_not_null())
ot_df = ot_df.with_columns([
    pl.col("OT Date").cast(pl.Date, strict=False),
    pl.col("Start Date Time").cast(pl.Datetime, strict=False),
    pl.col("End Date Time").cast(pl.Datetime, strict=False),
    pl.col("Duration").cast(pl.Float64, strict=False)
])

# =====================================================================
# [2/10] SHIFT DATE LOGIC (VNT & CA NIGHT)
# =====================================================================
print("[2/10] Allocating night shifts and identifying base shifts...")
ot_processed = ot_df.join(roster_key, left_on=["Emp ID", "OT Date"], right_on=["Emp ID", "Scheduled_Date"], how="left").with_columns(pl.col("Shift").fill_null("OFF"))
ot_processed = ot_processed.with_columns((pl.col("OT Date") - pl.duration(days=1)).alias("Prev OT Date"))
ot_processed = ot_processed.join(roster_key.rename({"Shift": "Prev_Day_Shift"}), left_on=["Emp ID", "Prev OT Date"], right_on=["Emp ID", "Scheduled_Date"], how="left").with_columns(pl.col("Prev_Day_Shift").fill_null("OFF"))

def check_night_shift(shift_col):
    split_shift = shift_col.str.split_exact("-", 1)
    has_dash = shift_col.str.contains("-").fill_null(False)
    start_num = split_shift.struct.field("field_0").str.strip_chars().cast(pl.Int32, strict=False).fill_null(0)
    end_num = split_shift.struct.field("field_1").str.strip_chars().cast(pl.Int32, strict=False).fill_null(2400)
    return pl.when(has_dash).then(end_num <= start_num).otherwise(pl.lit(False))

ot_processed = ot_processed.with_columns([
    pl.col("Start Date Time").dt.date().alias("otStartDate"),
    pl.col("Start Date Time").dt.hour().alias("otStartHour")
])

ot_processed = ot_processed.with_columns(
    pl.when(pl.col("otStartDate") > pl.col("OT Date")).then(pl.col("OT Date"))
    .when(pl.col("otStartDate") == pl.col("OT Date"))
    .then(
        pl.when(check_night_shift(pl.col("Prev_Day_Shift")) & (pl.col("otStartHour") < 14)).then(pl.col("OT Date") - pl.duration(days=1)).otherwise(pl.col("OT Date"))
    ).otherwise(pl.col("otStartDate")).alias("Shift Date")
)

ot_processed = ot_processed.with_columns(
    pl.when(pl.col("Shift Date") == pl.col("OT Date")).then(pl.col("Shift")).otherwise(pl.col("Prev_Day_Shift")).alias("Base Shift")
).with_columns(
    pl.when(pl.col("Base Shift").str.contains("-")).then(pl.lit("Regular OT")).otherwise(pl.lit("PO")).alias("OT Type")
)

# =====================================================================
# [3/10] CONSOLIDATING REGISTRATION & AUTO-GENERATING HOLIDAY OT
# =====================================================================
print("[3/10] Consolidating registrations and auto-generating Holiday entries...")
ot_processed = ot_processed.sort(["Emp ID", "Shift Date", "Start Date Time"])
ot_processed = ot_processed.with_columns(
    (pl.col("Start Date Time").dt.strftime("%H%M") + "-" + pl.col("End Date Time").dt.strftime("%H%M")).alias("OT_Range_Str")
)

reg_raw = ot_processed.clone()

reg_agg_exprs = [
    pl.col("OT Date").drop_nulls().unique().sort().cast(pl.String).str.join(", ").alias("OT Dates Combined"),
    pl.col("OT_Range_Str").drop_nulls().str.join(", ").alias("OT Range Register"),
    pl.col("Start Date Time").min().alias("_Reg_Min_Start"),
    pl.col("End Date Time").max().alias("_Reg_Max_End")
]

if "RTA Confirm" in ot_processed.columns: reg_agg_exprs.append(pl.col("RTA Confirm").drop_nulls().unique().sort().str.join(", ").alias("RTA Confirm Status"))
if "Email" in ot_processed.columns: reg_agg_exprs.append(pl.col("Email").first())
if "TL" in ot_processed.columns: reg_agg_exprs.append(pl.col("TL").first())
if "Name" in ot_processed.columns: reg_agg_exprs.append(pl.col("Name").first().alias("Agent Name"))
elif "Agent Name" in ot_processed.columns: reg_agg_exprs.append(pl.col("Agent Name").first())
if "Base Shift" in ot_processed.columns: reg_agg_exprs.append(pl.col("Base Shift").first())
if "OT Type" in ot_processed.columns: reg_agg_exprs.append(pl.col("OT Type").first())

reg_final = ot_processed.group_by(["Emp ID", "Shift Date"]).agg(reg_agg_exprs).with_columns(pl.lit(False).alias("Is_Holiday_OT"))

# --- HOLIDAY OT AUTO-GENERATION ---
# 1. Filter scheduled shifts on Holidays
holiday_roster = roster_melted.join(holidays_df, left_on="Scheduled_Date", right_on="Date", how="inner").filter(pl.col("Shift").str.contains("-"))

# 2. Enrich with Meta and Designation (SAFE SELECT)
meta_cols_to_get = [c for c in ["Emp ID", "Agent Name", "TL"] if c in roster_meta.columns]
holiday_roster = holiday_roster.join(roster_meta.select(meta_cols_to_get), on="Emp ID", how="left")
holiday_roster = holiday_roster.join(hc_ext, left_on=["Emp ID", "Scheduled_Date"], right_on=["OracleID", "Date"], how="left")

# Safely assign TL from Supervisor Name if available
if "Supervisor Name" in holiday_roster.columns:
    if "TL" in holiday_roster.columns:
        holiday_roster = holiday_roster.with_columns(pl.col("TL").fill_null(pl.col("Supervisor Name")))
    else:
        holiday_roster = holiday_roster.with_columns(pl.col("Supervisor Name").alias("TL"))
    holiday_roster = holiday_roster.drop("Supervisor Name", strict=False)
    
# 3. Anti-join with manual OT Register (Do NOT filter by Designation here so ALL get auto-gen rows)
holiday_roster = holiday_roster.join(ot_df, left_on=["Emp ID", "Scheduled_Date"], right_on=["Emp ID", "OT Date"], how="anti")

# 4. Parse Base Shift into Start/End Datetimes for Interval matching
split_shift = pl.col("Shift").str.split_exact("-", 1)
s_str = split_shift.struct.field("field_0").str.strip_chars().str.zfill(4)
e_str = split_shift.struct.field("field_1").str.strip_chars().str.zfill(4)
s_dt = pl.col("Scheduled_Date").dt.combine(s_str.str.strptime(pl.Time, "%H%M", strict=False))
e_time = e_str.str.strptime(pl.Time, "%H%M", strict=False)
e_dt = pl.when(e_time < s_str.str.strptime(pl.Time, "%H%M", strict=False)).then((pl.col("Scheduled_Date") + pl.duration(days=1)).dt.combine(e_time)).otherwise(pl.col("Scheduled_Date").dt.combine(e_time))

holiday_roster = holiday_roster.with_columns([s_dt.alias("Start Date Time"), e_dt.alias("End Date Time")])

# 5. Build and Concat Holiday Data
holiday_final = holiday_roster.select([
    pl.col("Emp ID"),
    pl.col("Scheduled_Date").alias("Shift Date"),
    pl.col("Shift").alias("Base Shift"),
    pl.lit("Regular OT").alias("OT Type"),
    pl.lit("PO (Present on Holiday)").alias("OT Dates Combined"),
    pl.col("Shift").alias("OT Range Register"),
    pl.col("Start Date Time").alias("_Reg_Min_Start"),
    pl.col("End Date Time").alias("_Reg_Max_End"),
    pl.lit("Holiday Auto-Gen").alias("RTA Confirm Status"),
    pl.col("Email"),
    pl.col("TL"),
    pl.col("Agent Name"),
    pl.lit(True).alias("Is_Holiday_OT")
])

holiday_raw = holiday_roster.select([
    pl.col("Emp ID"),
    pl.col("Email"),
    pl.col("Scheduled_Date").alias("Shift Date"),
    pl.col("Start Date Time"),
    pl.col("End Date Time")
])

# Combine standard registrations with Holiday auto-generations
common_cols_final = [c for c in reg_final.columns if c in holiday_final.columns]
reg_final = pl.concat([reg_final.select(common_cols_final), holiday_final.select(common_cols_final)], how="vertical")

common_cols_raw = [c for c in reg_raw.columns if c in holiday_raw.columns]
reg_raw = pl.concat([reg_raw.select(common_cols_raw), holiday_raw.select(common_cols_raw)], how="vertical")

# =====================================================================
# [4/10] PREPARING RAMCO DATA 
# =====================================================================
min_shift_date, max_shift_date = reg_final["Shift Date"].min(), reg_final["Shift Date"].max()
print(f"[4/10] Processing RAMCO data (Filtered: {min_shift_date} to {max_shift_date})...")

if os.path.exists(ramco_ot_csv_path):
    ramco_raw = pl.read_csv(ramco_ot_csv_path, encoding="utf-8", ignore_errors=True)
    ramco_raw = ramco_raw.with_columns([
        pl.col("Date").str.strptime(pl.Date, "%Y-%m-%d", strict=False),
        pl.col("EID").cast(pl.Int64, strict=False),
        pl.col("hours").cast(pl.Float64, strict=False)
    ])
    
    ramco_clean = ramco_raw.filter(
        pl.col("ot_type").str.starts_with("OT") & 
        (pl.col("hours") > 0) &
        (pl.col("Date") >= min_shift_date) & 
        (pl.col("Date") <= max_shift_date)
    ).group_by(["EID", "Date"]).agg([
        pl.col("hours").sum().alias("RAMCO_Logged_Hours"),
        pl.col("ot_status").first().alias("RAMCO_Status"),
        pl.col("ot_type").first().alias("OT Ramco Ratio Submitted")
    ])
else:
    ramco_clean = pl.DataFrame({"EID": [], "Date": [], "RAMCO_Logged_Hours": [], "RAMCO_Status": [], "OT Ramco Ratio Submitted": []}).with_columns([pl.col("EID").cast(pl.Int64), pl.col("Date").cast(pl.Date)])

# =====================================================================
# [5/10] OUTER JOIN (REGISTRATION vs RAMCO) - PARQUET INTEGRATION
# =====================================================================
print("[5/10] Performing Outer Join and mapping TL/Designation/LOB from Parquet...")
recon_df = reg_final.join(
    ramco_clean, left_on=["Emp ID", "Shift Date"], right_on=["EID", "Date"], how="outer"
).with_columns([
    pl.col("Emp ID").fill_null(pl.col("EID")),
    pl.col("Shift Date").fill_null(pl.col("Date")),
    pl.col("Is_Holiday_OT").fill_null(False) # Fill missing for RAMCO-only rows
])

recon_df = recon_df.join(roster_meta, on="Emp ID", how="left")

fill_meta_exprs = []
for col in ["Agent Name", "Email", "TL"]:
    col_r = f"{col}_right"
    if col_r in recon_df.columns:
        if col in recon_df.columns: fill_meta_exprs.append(pl.col(col).fill_null(pl.col(col_r)))
        else: fill_meta_exprs.append(pl.col(col_r).alias(col))
if fill_meta_exprs:
    recon_df = recon_df.with_columns(fill_meta_exprs)

# Extract Designation and LOB globally from Parquet
recon_df = recon_df.join(hc_ext, left_on=["Shift Date", "Emp ID"], right_on=["Date", "OracleID"], how="left")
if "Supervisor Name" in recon_df.columns:
    if "TL" in recon_df.columns: recon_df = recon_df.with_columns(pl.col("TL").fill_null(pl.col("Supervisor Name")))
    else: recon_df = recon_df.with_columns(pl.col("Supervisor Name").alias("TL"))
    recon_df = recon_df.drop("Supervisor Name", strict=False)

for c in ["Designation", "LOB"]:
    if c not in recon_df.columns: recon_df = recon_df.with_columns(pl.lit(None).cast(pl.String).alias(c))

recon_df = recon_df.with_columns([
    pl.col("RAMCO_Logged_Hours").fill_null(0),
    pl.col("RAMCO_Status").str.extract(r"-\s*([A-Za-z]+)", 1).alias("RAMCO_Approval_Status")
])

if "Shift" in roster_key.columns:
    recon_df = recon_df.join(roster_key.select(["Emp ID", "Scheduled_Date", "Shift"]), left_on=["Emp ID", "Shift Date"], right_on=["Emp ID", "Scheduled_Date"], how="left")
    if "Base Shift" in recon_df.columns: recon_df = recon_df.with_columns(pl.col("Base Shift").fill_null(pl.col("Shift")))
    else: recon_df = recon_df.with_columns(pl.col("Shift").alias("Base Shift"))
        
    if "OT Type" in recon_df.columns:
        recon_df = recon_df.with_columns(
            pl.when(pl.col("OT Type").is_null())
            .then(pl.when(pl.col("Base Shift").str.contains("-").fill_null(False)).then(pl.lit("Regular OT")).otherwise(pl.lit("PO")))
            .otherwise(pl.col("OT Type")).alias("OT Type")
        )
    recon_df = recon_df.drop("Shift", strict=False)

drop_cols = [c for c in recon_df.columns if c.endswith("_right")] + ["EID", "Date", "RAMCO_Status"]
recon_df = recon_df.drop([c for c in drop_cols if c in recon_df.columns])

# =====================================================================
# [6/10] WEEK & PST WEEK LOGIC
# =====================================================================
print("[6/10] Calculating Global PST Week...")

def calculate_pst_week(df):
    return df.with_columns([
        pl.col("_Reg_Min_Start").fill_null(pl.col("Shift Date").cast(pl.Datetime)).alias("_Base_TS")
    ]).with_columns([
        pl.when((pl.col("Shift Date") >= datetime(2026, 3, 8)) & (pl.col("Shift Date") < datetime(2026, 11, 1)))
        .then(pl.col("_Base_TS") - pl.duration(hours=14))
        .otherwise(pl.col("_Base_TS") - pl.duration(hours=15))
        .alias("_PST_TS")
    ]).with_columns([
        (pl.col("_PST_TS") - pl.duration(hours=3)).dt.truncate("1w").dt.date().alias("PST Week")
    ])

recon_df = recon_df.with_columns(pl.col("Shift Date").dt.truncate("1w").alias("Week"))
recon_df = calculate_pst_week(recon_df)

# =====================================================================
# [7/10] RTA & INTERVAL MATCHING (GRID SLOTS)
# =====================================================================
print("[7/10] Fetching RTA Data and matching Actual Productive through Grid Slots...")

rta_files = glob.glob(f"{rta_report_folder}/*.xlsx") + glob.glob(f"{rta_report_folder}/*.csv")
if rta_files:
    df_rta_list = [pl.read_excel(f, infer_schema_length=0) if f.endswith('.xlsx') else pl.read_csv(f, infer_schema_length=0, ignore_errors=True) for f in rta_files]
    RTA_REPORT = pl.concat(df_rta_list, how='diagonal').with_columns([
        pl.col("OracleID").cast(pl.Int64, strict=False),
        pl.col("Date_Converted").str.slice(0, 10).str.strptime(pl.Date, "%Y-%m-%d", strict=False),
        pl.col("sum_productive").cast(pl.Float64, strict=False),
    ]).rename({
        "Combined OT Range": "OT Range IEX"
    })
    recon_df = recon_df.join(RTA_REPORT.select(['Date_Converted','OracleID','Shift Tracking','OT PreShift','OT PostShift','OT Range IEX','start','end','sum_productive']), left_on=["Shift Date", "Emp ID"], right_on=["Date_Converted", "OracleID"], how="left")
else:
    recon_df = recon_df.with_columns(pl.lit(0.0).alias("sum_productive"))

interval_files = glob.glob(f"{rta_intervals_folder}/*.csv")
if interval_files:
    df_intervals = pl.concat([pl.read_csv(f, infer_schema_length=0, ignore_errors=True) for f in interval_files], how='diagonal')
    prod_int = df_intervals.filter(pl.col("Productive Aux Flag (Yes / No)") == "Yes").with_columns([
        pl.col("VNT_Intervals").str.strptime(pl.Datetime, "%Y-%m-%d %H:%M:%S", strict=False),
        pl.col("Duration").cast(pl.Float64, strict=False)
    ]).group_by(["Email Id", "VNT_Intervals"]).agg(pl.col("Duration").sum().alias("Interval_Productive"))

    ot_slots = reg_raw.select(["Emp ID", "Email", "Shift Date", "Start Date Time", "End Date Time"]).drop_nulls()
    ot_slots = ot_slots.with_columns(
        pl.datetime_ranges(start=pl.col("Start Date Time").dt.truncate("30m"), end=pl.col("End Date Time"), interval="30m", closed="left").alias("VNT_Intervals")
    ).explode("VNT_Intervals")
    
    ot_slots = ot_slots.with_columns((pl.col("VNT_Intervals") + pl.duration(minutes=30)).alias("Grid_Slot_End")).with_columns([
        pl.max_horizontal("VNT_Intervals", "Start Date Time").alias("S_Start"),
        pl.min_horizontal("Grid_Slot_End", "End Date Time").alias("S_End")
    ]).with_columns(((pl.col("S_End") - pl.col("S_Start")).dt.total_milliseconds() / 3_600_000).alias("S_Dur"))

    ot_agg = ot_slots.join(prod_int, left_on=["Email", "VNT_Intervals"], right_on=["Email Id", "VNT_Intervals"], how="left").group_by(["Emp ID", "Shift Date"]).agg([
        pl.col("S_Dur").sum().alias("OT Registered"),
        pl.col("Interval_Productive").sum().alias("OT Actual Productive")
    ])
    
    # APPLY HOLIDAY OVERRIDE FOR ACTUALS
    recon_df = recon_df.join(ot_agg, on=["Emp ID", "Shift Date"], how="left").with_columns([
        pl.when(pl.col("Is_Holiday_OT") == True).then(pl.col("sum_productive")).otherwise(pl.col("OT Registered")).fill_null(0).alias("OT Registered"),
        pl.when(pl.col("Is_Holiday_OT") == True).then(pl.col("sum_productive")).otherwise(pl.col("OT Actual Productive")).fill_null(0).alias("OT Actual Productive")
    ])
    recon_df = recon_df.with_columns(
        pl.max_horizontal(pl.col("sum_productive").fill_null(0) - pl.col("OT Actual Productive"), 0).alias("Base Shift Actual Productive")
    )
else:
    recon_df = recon_df.with_columns([
        pl.lit(0.0).alias("OT Registered"), pl.lit(0.0).alias("OT Actual Productive"), pl.lit(0.0).alias("Base Shift Actual Productive")
    ])

# =====================================================================
# [8/10] CALCULATE OT CONFIRM & CAPPING
# =====================================================================
print("[8/10] Computing OT Confirm and applying RAMCO/BACKEND Capping...")

recon_df = recon_df.with_columns([
    pl.col("OT Registered").alias("_TotalSlot"),
    pl.col("OT Actual Productive").alias("_TotalProductive")
])

recon_df = recon_df.with_columns([
    mround(pl.col("_TotalProductive"), 0.25).alias("_RoundedProd"),
    ((pl.col("_TotalSlot") / 4).floor() * 0.25).alias("_InitialBreak")
])

recon_df = recon_df.with_columns(pl.max_horizontal(pl.col("_TotalProductive") - pl.col("_InitialBreak"), 0).alias("_EffectiveHours"))

recon_df = recon_df.with_columns(
    pl.when(pl.col("_TotalProductive") < pl.col("_TotalSlot"))
    .then((pl.col("_EffectiveHours") / 4).floor() * 0.25)
    .otherwise(pl.col("_InitialBreak")).alias("_FinalBreak")
)

recon_df = recon_df.with_columns([
    mround(pl.col("_RoundedProd") + pl.col("_FinalBreak"), 0.5).alias("_FinalRounded"),
    (pl.col("_TotalSlot") - pl.when(pl.col("_TotalSlot") >= 9).then(1).otherwise(0)).alias("_MaxPayable")
])

# >> UPDATE: Logic OT Confirm cho Advisor I/CS trong ngày Lễ
holiday_confirm_expr = pl.when(pl.col("sum_productive").fill_null(0.0) >= 7.0).then(
    8.0
).otherwise(
    mround(pl.col("sum_productive").fill_null(0.0), 0.5)
)

recon_df = recon_df.with_columns(
    pl.when(
        (pl.col("Is_Holiday_OT") == True) & 
        (pl.col("Designation").is_in(["Advisor I", "Customer Service", "Advisor I, Customer Service"]))
    )
    .then(holiday_confirm_expr)
    .otherwise(
        pl.when((pl.col("_TotalProductive") == 0) | pl.col("_TotalProductive").is_null()).then(0)
        .otherwise(pl.min_horizontal(pl.col("_FinalRounded"), pl.col("_MaxPayable")))
    ).alias("OT Confirm")
)

recon_df = recon_df.with_columns([
    pl.when((pl.col("OT Type") == "PO") | (pl.col("Is_Holiday_OT") == True))
    .then(pl.min_horizontal(pl.col("OT Confirm"), 12))
    .when(pl.col("OT Type") == "Regular OT")
    .then(pl.min_horizontal(pl.col("OT Confirm"), 4))
    .otherwise(0).alias("RAMCO OT") 
])
recon_df = recon_df.with_columns((pl.col("OT Confirm") - pl.col("RAMCO OT")).alias("BACKEND OT"))

# =====================================================================
# [9/10] CLEAN UP TEMPORARY METRICS AND REORDER COLUMNS
# =====================================================================
print("[9/10] Cleaning up metrics and reordering columns...")
cols_to_drop = [c for c in recon_df.columns if c.startswith("_")] + ["Date_Converted", "OracleID", "Is_Holiday_OT"]
recon_df = recon_df.drop([c for c in cols_to_drop if c in recon_df.columns])

first_cols = ["Week", "PST Week"]
first_cols = [c for c in first_cols if c in recon_df.columns]
other_cols = [c for c in recon_df.columns if c not in first_cols]
recon_df = recon_df.select(first_cols + other_cols)

# =====================================================================
# [10/10] EXPORT FINAL REPORT
# =====================================================================
print("[10/10] Exporting final report...")
recon_df = recon_df.sort(["Shift Date", "Emp ID"])
recon_df.write_excel(output_ot_path)

print("-" * 60)
print(f"DONE! Data successfully exported to:\n>> {output_ot_path}")
print("-" * 60)

preview_cols = [
    "Emp ID", "Agent Name", "Week", "PST Week", "RTA Confirm Status",
    "OT Range Register", "OT Registered", "OT Actual Productive", "OT Confirm", 
    "RAMCO_Logged_Hours", "OT Ramco Ratio Submitted", "RAMCO_Approval_Status"
]
print(recon_df.select([c for c in preview_cols if c in recon_df.columns]).head(10))

C:\Users\huuchinh.nguyen\AppData\Local\Temp\ipykernel_25172\2767906593.py:46: DataOrientationWarning: Row orientation inferred during DataFrame construction. Explicitly specify the orientation by passing `orient="row"` to silence this warning.
  holidays_df = pl.DataFrame(holidays_data, schema=["Date", "Holiday_Name"]).with_columns(pl.col("Date").str.strptime(pl.Date, "%Y-%m-%d"))


STARTING RECONCILIATION REPORT PROCESSING (HOLIDAY AUTO-GEN INCLUDED)...
------------------------------------------------------------
[1/10] Processing Roster, HC Parquet, and OT Plan data...


Could not determine dtype for column 18, falling back to string
Could not determine dtype for column 19, falling back to string
C:\Users\huuchinh.nguyen\AppData\Local\Temp\ipykernel_25172\2767906593.py:246: DeprecationWarning: use of `how='outer'` should be replaced with `how='full'`.
(Deprecated in version 0.20.29)
  recon_df = reg_final.join(


[2/10] Allocating night shifts and identifying base shifts...
[3/10] Consolidating registrations and auto-generating Holiday entries...
[4/10] Processing RAMCO data (Filtered: 2026-02-16 to 2026-05-11)...
[5/10] Performing Outer Join and mapping TL/Designation/LOB from Parquet...
[6/10] Calculating Global PST Week...
[7/10] Fetching RTA Data and matching Actual Productive through Grid Slots...
[8/10] Computing OT Confirm and applying RAMCO/BACKEND Capping...
[9/10] Cleaning up metrics and reordering columns...
[10/10] Exporting final report...
------------------------------------------------------------
DONE! Data successfully exported to:
>> C:/Users/huuchinh.nguyen/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/OUTPUT_OT_REGISTRATION_REPORT.xlsx
------------------------------------------------------------
shape: (10, 12)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ Emp ID    ┆ Agent     ┆ Week      ┆ PST Week 